In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.dates as mdates
from sklearn.metrics import r2_score

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res

def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft


In [ ]:
def line_fit(X,Y):
    n = len(X)
    P, residuals, rank, singular_values, rcond= np.polyfit(X,Y,1, full=True)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.2, np.max(X)+dx*0.2])
    Y_new = np.polyval(P, X_new)
    rmse = np.sqrt(residuals[0] / n)
    #Create a polynomial function from the coefficients
    pol = np.poly1d(P)
    Y_Pred = pol(x)
    r_squared = r2_score(Y, Y_Pred)
    return X_new, Y_new, P, rmse, r_squared

In [ ]:
def exponential_func(x, a, b, c):
    return a * np.exp(b * x) + c

def exp_fit(X,Y, P0=np.array([1, 1e-6, 1])):
    """returns  X_new, Y_new, popt, pcov"""
    popt, pcov = curve_fit(exponential_func, X,Y, p0=P0)
    dx = np.max(X) - np.min(X)
    X_new = np.linspace( np.min(X)-dx*0.2, np.max(X)+dx*0.2, 100)    
    Y_new = exponential_func(X_new, *popt)
    return  X_new, Y_new, popt, pcov

In [ ]:
%run ../2026-02-03_Age-Model/html_peak_match/20260323_read_data.py
df_sorted

In [ ]:
df[df['age_final']>1995]

In [ ]:
climate = pd.read_csv('../2026-01-15_Climate-Model/analysis/climate_df.csv')
climate['datetime'] = pd.to_datetime(climate['datetime'])
climate_month = climate.groupby(climate['datetime'].dt.month).mean()
climate_month

In [ ]:
data_old = pd.read_excel('20251126_DepthLaurenPaper/CoralDepthData.xlsx')
data_old.rename(columns={data_old.columns[0]:'id', data_old.columns[1]:'d13c', data_old.columns[2]:'d18o'}, inplace=True)
data_old['path'] = [x[1] for x in data_old['id'].str.split(' ')]
data_old['idx'] = [x.replace('A','') for x in [x[-1] for x in data_old['id'].str.split(' ')]]
data_old['idx'] = pd.to_numeric(data_old['idx'])
depths = np.array([-105, -95, -68, -58, -48, -37, -27]) - 60
#depths = np.array([-105, -95, -73, -63, -53, -42, -32.5]) - 60
paths = np.unique(data_old['path'].values)
path_depth = dict(zip(paths,depths))
data_old['depth'] = data_old['path'].map(path_depth)
data_old

In [ ]:
data = pd.read_excel("2024-depth-ABCDE_memo-edited_Agemodeled.xlsx")
data['path'] = [x[0][0] for x in data['Identifier 1'].str.split(' ')]
data['idx'] = [int(str.replace(x[1],'A','')) for x in data['Identifier 1'].str.split(' ')]
data[['d13c_ma', 'd18o_ma']] = data[['path','d13c','d18o']].groupby('path').rolling(3, center=True).mean().values
data.loc[(data['path'] == 'C')&(data['idx']==14), 'd13c'] = np.nan
offset = 60
paths = data['path'].dropna().unique()
depths = np.array([-100, -94, -70, -58.5, -48])-offset
#depth_dict = {'A':-100, 'B':-94, 'C':-70, 'D':-58.5, 'E':-48}
depth_dict = dict(zip(paths, depths))
data['depth'] = data['path'].map(depth_dict)
data#.iloc[70:80]

In [ ]:
data.groupby('path')['idx'].max().mean()

In [ ]:
data_old.groupby('path')['idx'].max().mean()

In [ ]:
window_size = 3
window_size2 = 36

fred = pd.read_excel('../2025-08-22_FredsData-post2008/FredData_editedMemo.xlsx').dropna()

fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['d13c_ma','d18o_ma']] = fred[['d13c','d18o_pdb']].rolling(window_size, center=True).mean()
fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
#fred.to_csv('fred_pdb.csv')
fred_resampled = fred.drop(fred.columns[3:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['decimal_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred_resampled

In [ ]:
paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

for p in paths:

    data_iter = data[data['path'] == p]
    ax.plot(data_iter['idx'] , data_iter['d18o_ma'] ,".-", label = "Depth "+p)
ax.legend()
ax.set_title(r"$\delta18$O")
fig.savefig("d18O.pdf")

In [ ]:
paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

for p in paths:

    data_iter = data[data['path'] == p]
    ax.plot(data_iter['idx'] , data_iter['d13c'] ,".-", label = "Depth "+p)
ax.legend()
ax.grid()

ax.set_title(r"$\delta13$C")
fig.savefig("d13C.pdf")

In [ ]:
paths = data_old['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

for p in paths:

    data_iter = data_old[data_old['path'] == p]
    ax.plot(data_iter['idx'] , data_iter['d13c'] ,".-", label = "Depth "+p)
ax.legend()
ax.grid()

ax.set_title(r"$\delta13$C")
#fig.savefig("d13C.pdf")

In [ ]:
data_old

In [ ]:
data.to_csv('depth_data_memo.csv')


In [ ]:
data_ = data

#depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap((depths/depths.min())**2 )

colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5']

paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

paths_r = paths[::-1]

for i,p in enumerate(paths_r[1:]):

    c = colors[i]
    data_iter = data[data['path'] == p]
    ax.plot(data_iter['age_absolute'] , data_iter['d18o'] ,".", color=c, label = "Depth "+p)
    ax.plot(data_iter['age_absolute'].rolling(3).mean() , data_iter['d18o'].rolling(3).mean(), color=c)

fred_selected = fred[(fred['decimal_year']>1995) & (fred['decimal_year']<2000)]
ax.plot(fred_selected['decimal_year'], fred_selected['d18o_pdb'], color='red', label='fred')

ax.plot(climate['age_absolute'], climate['synthetic_d18o']+25)

ax.legend()
ax.grid()

ax.set_xlim(1995, 2000)

ax.set_title(r"$\delta18$O")
#fig.savefig("d13C.png")
#fig.savefig("d13C.pdf")

In [ ]:
data_ = data

#depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap((depths/depths.min())**2 )

colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5']

paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

paths_r = paths

for i,p in enumerate(paths_r):

    c = colors[i]
    data_iter = data[data['path'] == p]
    ax.plot(data_iter['age_absolute'] , data_iter['d18o'] ,".", color=c, label = "Depth "+p)
    ax.plot(data_iter['age_absolute'].rolling(3).mean() , data_iter['d18o'].rolling(3).mean(), color=c)

fred_selected = fred[(fred['decimal_year']>1995) & (fred['decimal_year']<2000)]
ax.plot(fred_selected['decimal_year'], fred_selected['d18o_pdb'], color='red', label='fred')

ax.plot(climate['age_absolute'], climate['synthetic_d18o']+25)

ax.legend()
ax.grid()

ax.set_xlim(1995, 2000)

ax.set_title(r"$\delta18$O")
#fig.savefig("d13C.png")
#fig.savefig("d13C.pdf")

In [ ]:
data_ = data

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap((depths/depths.min())**2 )

colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5']

paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

paths_r = paths[::-1]

for i,p in enumerate(paths_r[1:]):

    c = colors[i]
    data_iter = data[data['path'] == p]
    ax.plot(data_iter['age_absolute'] , data_iter['d13c'] ,".", color=c, label = "Depth "+p)
    ax.plot(data_iter['age_absolute'] , data_iter['d13c'].rolling(3).mean(), color=c)

fred_selected = fred[(fred['decimal_year']>1995) & (fred['decimal_year']<2000)]
ax.plot(fred_selected['decimal_year'], fred_selected['d13c_ma'], color='red', label='fred')


ax.legend()
ax.grid()

ax.set_title(r"$\delta13$C")
#fig.savefig("d13C.png")
#fig.savefig("d13C.pdf")

In [ ]:
data_ = data

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap((depths/depths.min())**2 )

colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5']

paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

paths_r = paths

for i,p in enumerate(paths_r):

    c = colors[i]
    data_iter = data[data['path'] == p]
    ax.plot(data_iter['age_absolute'] , data_iter['d13c'] ,".", color=c, label = "Depth "+p)
    ax.plot(data_iter['age_absolute'] , data_iter['d13c'].rolling(3).mean(), color=c)

fred_selected = fred[(fred['decimal_year']>1995) & (fred['decimal_year']<2000)]
ax.plot(fred_selected['decimal_year'], fred_selected['d13c_ma'], color='red', label='fred')


ax.legend()
ax.grid()

#ax.set_title(r"$\delta13$C")
#fig.savefig("d13C.png")
#fig.savefig("d13C.pdf")

In [ ]:
data_selected = data[~data['ref_age'].isna()]

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap((depths/depths.min())**2 )

colors = ['#F58727', '#F5EE27', '#95F527', '#2795F5', '#8727F5']

paths = data['path'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

paths_r = paths

for i,p in enumerate(paths_r):

    c = colors[i]
    data_iter = data_selected[data_selected['path'] == p]
    ax.plot(data_iter['age_absolute'] , data_iter['d13c'] ,".", color=c, label = "Depth "+p)
    #ax.plot(data_iter['age_absolute'] , data_iter['d13c'].rolling(3).mean(), color=c)

#fred_selected = fred[(fred['decimal_year']>1995) & (fred['decimal_year']<2000)]
#ax.plot(fred_selected['decimal_year'], fred_selected['d13c_ma'], color='red', label='fred')


ax.legend()
ax.grid()

#ax.set_title(r"$\delta13$C")
#fig.savefig("d13C.png")
#fig.savefig("d13C.pdf")

In [ ]:
#data_ = data[(data['path'] != 'E') & (data['path'] != 'C')]
data_ = data_selected
#data_ = data
depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(5, 5))

for i,p in enumerate(paths):
    
    data_iter = data_[data_['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')







fred_selected['depth'] = -145
data_iter = fred_selected
ax.plot(data_iter['d13c'],data_iter['depth'], '.', color='blue')
ax.errorbar(x=data_iter['d13c'].mean(), 
            y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
            xerr=data_iter['d13c'].std(), ecolor=c,
            capsize=10)
            #label='mean')
ax.text(-0.9, depth_dict[p]-2, p, color=c)





ax.legend()
ax.set_ylim(-180, -110) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -110, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

#fig.savefig('depth_d13c_missing_E_C.pdf', bbox_inches='tight', pad_inches=0.15)
#fig.savefig('depth_d13c_missing_E_C.png', bbox_inches='tight', pad_inches=0.15)

In [ ]:
data_ = data[(data['age_absolute']>1996.5) & (data['age_absolute']<1999)]
#data_ = data_[data_['path'] != 'E']
depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data_[data_['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')

ax.legend()
ax.set_ylim(-180, -90) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -90, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

#fig.savefig('depth_d13c.pdf')
#fig.savefig('depth_d13c_aged.png')

In [ ]:
pd.concat([df1[common_cols], df2[common_cols]], ignore_index=True)

In [ ]:
fred_selected

In [ ]:
fred_selected['Depth'] = -145
data_iter = fred_selected

In [ ]:
#data_ = data[(data['path'] != 'E') & (data['path'] != 'C')]
data_ = data[(data['path'] != 'E')]
#data_ = data
depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(5, 5))

for i,p in enumerate(paths):
    
    data_iter = data_[data_['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')







fred_selected['depth'] = -145
data_iter = fred_selected
ax.plot(data_iter['d13c'],data_iter['depth'], '.', color='blue')
ax.errorbar(x=data_iter['d13c'].mean(), 
            y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
            xerr=data_iter['d13c'].std(), ecolor=c,
            capsize=10)
            #label='mean')
ax.text(-0.9, depth_dict[p]-2, p, color=c)





ax.legend()
ax.set_ylim(-180, -110) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -110, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

fig.savefig('depth_d13c_missing_E_C.pdf', bbox_inches='tight', pad_inches=0.15)
fig.savefig('depth_d13c_missing_E_C.png', bbox_inches='tight', pad_inches=0.15)

In [ ]:
data_ = data_[data_['path'] != 'E']
depths = data['depth'].unique()
paths = data['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(5, 5))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',ms=10,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')




fred_selected['depth'] = -145
data_iter = fred_selected
ax.plot(data_iter['d13c'],data_iter['depth'], '.', color='blue')
ax.errorbar(x=data_iter['d13c'].mean(), 
            y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
            xerr=data_iter['d13c'].std(), ecolor=c,
            capsize=10)
            #label='mean')
ax.text(-0.9, -145-2, 'Prior'+'\n'+'Results', color='blue')







ax.legend()
ax.set_ylim(-180, -90)
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -90, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

fig.savefig('depth_d13c_.pdf', bbox_inches='tight', pad_inches=0.15)
fig.savefig('depth_d13c_.png', bbox_inches='tight', pad_inches=0.15)

In [ ]:
#data_ = data[(data['path'] != 'E') & (data['path'] != 'C')]
data_ = data[(data['path'] != 'E')]
#data_ = data
depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(5, 5))

for i,p in enumerate(paths):
    
    data_iter = data_[data_['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')



ax.legend()
ax.set_ylim(-180, -110) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -110, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

fig.savefig('depth_d13c_missing_E_C.pdf', bbox_inches='tight', pad_inches=0.15)
fig.savefig('depth_d13c_missing_E_C.png', bbox_inches='tight', pad_inches=0.15)

In [ ]:
data

In [ ]:
data_H = df_sorted[(df_sorted['path_name']=='H') & 
    (df_sorted['age_final']>1995) & 
    (df_sorted['age_final']<2000)][['d13c', 'd18o', 'd13c_ma',
                                    'd18o_ma','age_final', 'path_name', 'path_num']]
data_H = data_H.rename(columns={'path_name':'path', 'path_num':'idx'})
data_H['depth'] = -36 -60
data_H

In [ ]:
#data_ = data[data['path'] != 'E']
data_ = data

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.8, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, '--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')

ax.plot(data_H['d13c'],data_H['depth'], '.', color='blue')
ax.errorbar(x=data_H['d13c'].mean(), 
            y=np.unique(data_H['depth']), yerr=0, fmt='s',color='blue',ms=12,
            xerr=data_H['d13c'].std(), ecolor=c,
            capsize=10)
ax.text(-0.8, np.unique(data_H['depth'])-4, 'Second'+'\n'+'Campaign', color='blue')



fred_selected['depth'] = -145
data_iter = fred_selected
ax.plot(data_iter['d13c'],data_iter['depth'], '.', color='blue')
ax.errorbar(x=data_iter['d13c'].mean(), 
            y=np.unique(data_iter['depth']), yerr=0, fmt='s',color='blue',ms=12,
            xerr=data_iter['d13c'].std(), ecolor=c,
            capsize=10)
ax.text(-0.9, np.unique(data_iter['depth'])-2, 'First'+'\n'+'Campaign', color='blue')
#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p].copy()
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='v',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old.copy()
x = data_iter.groupby('path')['d13c'].mean().values
y = data_iter['depth'].unique()
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')

#ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Mean Point Fit')

#fig.savefig('depth_d13c_compare.pdf')

In [ ]:
#data_ = data[data['path'] != 'E']
data_ = data

depths = data_['depth'].unique()
paths = data_['path'].unique()
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p].copy()
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(0.2, depth_dict[p]-2, p, color=c)
x = data_.groupby('path')['d13c'].mean().values
y = depths
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, '--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')

ax.plot(data_H['d13c'],data_H['depth'], '.', color='blue')
ax.errorbar(x=data_H['d13c'].mean(), 
            y=np.unique(data_H['depth']), yerr=0, fmt='s',color='blue',ms=12,
            xerr=data_H['d13c'].std(), ecolor=c,
            capsize=10)
ax.text(0.2, np.unique(data_H['depth'])-4, 'Second'+'\n'+'Campaign', color='blue')



fred_selected['depth'] = -145
data_iter = fred_selected
ax.plot(data_iter['d13c'],data_iter['depth'], '.', color='blue')
ax.errorbar(x=data_iter['d13c'].mean(), 
            y=np.unique(data_iter['depth']), yerr=0, fmt='s',color='blue',ms=12,
            xerr=data_iter['d13c'].std(), ecolor=c,
            capsize=10)
ax.text(0.1, np.unique(data_iter['depth'])-2, 'First'+'\n'+'Campaign', color='blue')
#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p].copy()
    data_iter['d13c'] = data_iter['d13c']+0.3
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='v',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(0.1, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old.copy()
data_iter['d13c'] = data_iter['d13c']+0.3
x = data_iter.groupby('path')['d13c'].mean().values
y = data_iter['depth'].unique()
X, Y, P, rmse, r2= line_fit(x,y)

ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+
        f'RMSE={rmse:.3f}'+'\n'+
       f'$R^2$={r2:.3f}')

#ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-3.5, 0)
ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Mean Point Fit')

#fig.savefig('depth_d13c_compare.pdf')

In [ ]:
# Calculate the maximum path_num for each unique path ('A', 'B', 'C', etc.)
# This ensures the linear scale is correct for paths with different numbers of samples.
path_max_nums = data_old.groupby('path')['idx'].transform('max')

# Apply the linear interpolation formula for each row:
# age = StartYear + (CurrentNum - 1) * (TotalSpan / (MaxNum - 1))
# We use.replace(0, 1) to handle any paths that might contain only a single sample.
data_old['age'] = 2004 + (data_old['idx'] - 1) * (2006.5 - 2004) / (path_max_nums - 1).replace(0, 1)

# Display the first few rows to verify the assignment
print(data_old[['path', 'idx', 'age']].head())

In [ ]:
data_old

In [ ]:
# 1. Standardize and prepare each dataset
# New Data (from agemodel_pathsorted.xlsx)
df_new = data.copy()
df_new['label'] = 'New Data - ' + df_new['path'].astype(str)
df_new['age'] = df_new['age_absolute']
df_new['drill'] = '95-00'
df_new = df_new[['d13c', 'depth', 'age', 'label', 'path', 'drill']]

# Second Campaign (Path H)
df_h = data_H.copy()
df_h['label'] = 'Second Campaign H'
df_h['age'] = df_h['age_final']
df_h['path'] = 'H*'
df_h['drill'] = '95-00'
df_h = df_h[['d13c', 'depth', 'age', 'label', 'path', 'drill']]

# First Campaign (Fred Data)
df_fred = fred_selected.copy()
df_fred['label'] = 'First Campaign'
df_fred['depth'] = -145  # Assigning the constant depth used in your plot
df_fred['age'] = df_fred['decimal_year']
df_fred['path'] = 'F*'
df_fred['drill'] = '05-06'
df_fred = df_fred[['d13c', 'depth', 'age', 'label', 'path', 'drill']]

# Old Data (ABCDE) - applying the +0.3 shift used in the plot
df_old = data_old.copy()
df_old['d13c'] = df_old['d13c'] + 0.25
df_old['label'] = 'Old Data - ' + df_old['path'].astype(str)
df_old['drill'] = '05-06'
df_old = df_old[['d13c', 'depth', 'age', 'label', 'path', 'drill']]

# 2. Merge all into a single dataframe
merged_df = pd.concat([df_new, df_h, df_fred, df_old], ignore_index=True)

# 3. Calculate mean d13c for each unique depth
depth_means = merged_df.groupby('depth')['d13c'].mean().reset_index()
merged_df['depth_norm'] = (merged_df['depth']-merged_df['depth'].min())/(merged_df['depth'].max() - merged_df['depth'].min())

# 4. Perform line fits and compare
# Fit A: Over all raw points
#x_all = merged_df['d13c'].values
#y_all = merged_df['depth'].values
#X_all, Y_all, P_all, rmse_all, r2_all = line_fit(x_all, y_all)

# Fit B: Over the averaged depth points
#x_mean = depth_means['d13c'].values
#y_mean = depth_means['depth'].values
#X_mean, Y_mean, P_mean, rmse_mean, r2_mean = line_fit(x_mean, y_mean)

# 5. Summary Results
#print("--- Fit Comparison ---")
#print(f"Fit over ALL points:    y = {P_all:.4f}x + {P_all[1]:.4f} (R² = {r2_all:.4f})")
#print(f"Fit over DEPTH MEANS:   y = {P_mean:.4f}x + {P_mean[1]:.4f} (R² = {r2_mean:.4f})")

merged_df

In [ ]:
data_types

In [ ]:
#data_ = data[data['path'] != 'E']
data_ = merged_df

depths = data_['depth'].unique()
paths = data_['label'].unique()

viridis_cmap = plt.get_cmap('viridis')
hot_cmap = plt.get_cmap('inferno_r')
cool_cmap = plt.get_cmap('cool')
twilight_cmap = plt.get_cmap('twilight')


data_types = np.unique(merged_df['drill'])
# '2021', '2024', 'First Campaign', 'Second Campaign'

#cmap_dict= dict(zip(data_types, [cool_cmap,viridis_cmap, hot_cmap, twilight_cmap]))
#marker_dict = dict(zip(data_types, ['s', '^', 'v', 's']))
#xpos_dict = dict(zip(data_types, [0, 0, 0.4, 0.4])) 

cmap_dict= dict(zip(data_types, [hot_cmap, viridis_cmap, cool_cmap, twilight_cmap]))
marker_dict = dict(zip(data_types, ['^', 'v','s','s']))
xpos_dict = dict(zip(data_types, [0,-2.6, 0, -2.6])) 


fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = merged_df[merged_df['label'] == p].copy()
    drill_iter = np.unique(data_iter['drill'])[0]
    c = cmap_dict[drill_iter](data_iter['depth_norm'].mean())
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt=marker_dict[drill_iter],color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.8+xpos_dict[drill_iter], np.unique(data_iter['depth']), 
            #drill_iter+'\n'+np.unique(data_iter['path'])[0], color=c,backgroundcolor='#F2F2F2' )
            np.unique(data_iter['path'])[0], color=c,backgroundcolor='#F2F2F2' )
        
x = data_.groupby('label')['d13c'].mean().values
y = data_.groupby('label')['depth'].mean().values
X, Y, P, rmse, r2= line_fit(x,y)

#ax.plot(X,Y, '--', color='g', label='Whole data'+'\n'+f'y = {P[0]:.1f}x+{P[1]:.1f}'+'\n'+
ax.plot(X,Y, '--', color=c, label='All Data'+'\n'+f'n = {P[0]:.1f}'+'\n'+f'$R^2$={r2:.1f}')

    # f'RMSE={rmse:.3f}'+'\n'+
#       f'$R^2$={r2:.1f}')

for types in data_types:
    data_typ = data_[data_['drill']==types]
    x = data_typ.groupby('label')['d13c'].mean().values
    y = data_typ.groupby('label')['depth'].mean().values
    X, Y, P, rmse, r2= line_fit(x,y)

    c = cmap_dict[types](data_typ['depth_norm'].mean())
    #ax.plot(X,Y, '--', color=c, label=types+' data'+'\n'+f'y = {P[0]:.1f}x{P[1]:.1f}'+'\n'+
    ax.plot(X,Y, '--', color=c, label=types+' Data'+'\n'+f'n = {P[0]:.1f}'+'\n'+f'$R^2$={r2:.1f}')
           # f'RMSE={rmse:.1f}'+'\n'+
           #f'$R^2$={r2:.1f}')
    

ax.legend(loc='upper center', ncols=3)
ax.set_ylim(-180, -60) 
ax.set_xlim(-3.5, -0.5)
ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Relative Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
#ax.set_title('')

#------------------------------------------------------------------------------------

fig.savefig('depth_d13c_wholedata.pdf')

In [ ]:
paths = ['A','B','C','D','E']
data_selected = data[data['path'].isin(paths)]
depths = data_selected['depth'].unique()

viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)

data_iter = data[data['path'].isin(paths)].dropna()
x = data_iter['d13c'].values
y = data_iter['depth'].values
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y,'--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')


#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old

x = data_iter['d13c'].values
y = data_iter['depth'].values
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Complete Data Fit')

fig.savefig('depth_d13c_compare2.pdf')

In [ ]:
def exponential_func(x, a, b, c):
    return a * np.exp(b * x) + c

def exp_fit(X,Y, P0=np.array([1, 1, 1])):
    """returns  X_new, Y_new, popt, pcov"""
    popt, pcov = curve_fit(exponential_func, X,Y, p0=P0, maxfev=20000)
    dx = np.max(X) - np.min(X)
    X_new = np.linspace( np.min(X)-dx*0.2, np.max(X)+dx*0.2, 100)    
    Y_new = exponential_func(X_new, *popt)
    return  X_new, Y_new, popt, pcov

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

def exponential_func(x, a, b, c):
    return a * np.exp(b * x) + c

def guess_initial_params(X, Y):
    X = np.asarray(X, float)
    Y = np.asarray(Y, float)
    
    # Guess c as something slightly below the minimum of Y
    c0 = Y.min() - 0.1 * abs(Y.min())
    
    # Shift Y so it's (hopefully) positive for log-transform
    Y_shifted = Y - c0
    Y_shifted[Y_shifted <= 0] = np.min(Y_shifted[Y_shifted > 0]) * 0.5
    
    # Fit log(Y - c0) ~ log(a0) + b0 * X (linear regression)
    b0, loga0 = np.polyfit(X, np.log(Y_shifted), 1)
    a0 = np.exp(loga0)
    
    return np.array([a0, b0, c0])

def exp_fit(X, Y, P0=None):
    """returns  X_new, Y_new, popt, pcov"""
    X = np.asarray(X, float)
    Y = np.asarray(Y, float)
    
    if P0 is None:
        P0 = guess_initial_params(X, Y)
    
    popt, pcov = curve_fit(
        exponential_func,
        X,
        Y,
        p0=P0,
        maxfev=20000
    )
    
    dx = np.max(X) - np.min(X)
    X_new = np.linspace(np.min(X) - dx * 0.2, np.max(X) + dx * 0.2, 100)
    Y_new = exponential_func(X_new, *popt)
    return X_new, Y_new, popt, pcov


In [ ]:
paths = ['A','B','C','D','E']
data_selected = data[data['path'].isin(paths)]
depths = data_selected['depth'].unique()

viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, depth_dict[p]-2, p, color=c)

data_iter = data[data['path'].isin(paths)].dropna()
x = data_iter['d13c'].values
y = data_iter['depth'].values
#X, Y, P, rmse= line_fit(x,y)

#ax.plot(X,Y,'--', color='g', label='New data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')
#X, Y, P, pcov= exp_fit(x,y, P0=np.array([50.0, 0.5, -150.0]))
X, Y, P, pcov= exp_fit(x,y)
ax.plot(X,Y, '--', color='g', label='New data')

#------------------------------------------------------------------------------------

#paths = ['A','C','D']
paths = data_old['path'].unique()
depths = data_old['depth'].unique()


viridis_cmap = plt.get_cmap('hot_r')
colors = viridis_cmap(depths/depths.min()-0.1)

for i,p in enumerate(paths):
    
    data_iter = data_old[data_old['path'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='^',color=c,ms=12,
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

#data_iter = data_old[data_old['path'].isin(paths)]
data_iter = data_old

x = data_iter['d13c'].values
y = data_iter['depth'].values
#X, Y, P, pcov= exp_fit(x,y, P0=np.array([ 3.40573857e+04,  9.08829981e-04, -3.41132214e+04]))
X, Y, P, pcov= exp_fit(x,y)

#ax.plot(X,Y, '--', color='r', label='Old data'+'\n'+f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')
ax.plot(X,Y, '--', color='r', label='Old data')


                       
#x = np.linspace(-4,0,100)
#y=exponential_func(x , 7000,2.4,-165)
#ax.plot(x,y, '--', color='r', label='Old data')

#ax.legend(loc='upper left')
ax.set_ylim(-180, -60) 
ax.set_xlim(-4, 0)
#ax.set_yticks(np.arange(-180, -50, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Complete Data Fit')

#fig.savefig('depth_d13c_compare2.pdf')

In [ ]:
data_old.to_csv('20251126_DepthLaurenPaper/data_depth.csv')

In [ ]:
fig, ax = plt.subplots(1, figsize=(8,5))

for g in ["A","D"]:
    
    ax.plot(locals()["data_"+g]["Id_num"], locals()["data_"+g].iloc[:,5],".-", label = "Depth "+g)
ax.legend()
ax.set_title(r"$\delta13C$")
fig.savefig("d13C.pdf")